In [ ]:
import requests
import json
import time
import os

In [ ]:
import zipfile
import json
import pandas as pd
from io import BytesIO

URL = "https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip"
print(f"Скачиваем {URL} (23 МБ)...")
response = requests.get(URL)
print(f"Загружено: {len(response.content):,} байт\n")

with zipfile.ZipFile(BytesIO(response.content)) as zf:
    files = zf.namelist()

    # ===== ДИАГНОСТИКА =====
    print("=" * 60)
    print("ДИАГНОСТИКА: ИЩЕМ DJANGO")
    print("=" * 60)

    # Ищем разными способами
    django_exact = [f for f in files if 'django' in f.lower()]
    django_json = [f for f in files if 'django' in f.lower() and f.endswith('.json')]

    print(f"Всего файлов в архиве: {len(files):,}")
    print(f"Файлов с 'django' в названии: {len(django_exact)}")
    print(f"Из них .json: {len(django_json)}")

    if django_json:
        print("\nПримеры django-файлов:")
        for f in django_json[:5]:
            print(f"  {f}")
    else:
        print("\nСтранно, django не найден. Пробуем поиск по ID...")
        # Может django-уязвимости записаны как GHSA/PYSEC/MAL?
        django_in_id = []
        for f in files[:500]:  # Проверим первые 500 имён
            # Читаем содержимое и проверяем id или affected
            with zf.open(f) as fh:
                try:
                    data = json.loads(fh.read())
                    affected_pkgs = [a.get('package', {}).get('name', '') for a in data.get('affected', [])]
                    if any('django' in pkg.lower() for pkg in affected_pkgs):
                        django_in_id.append((f, affected_pkgs))
                except:
                    pass
        print(f"В первых 500 файлах найдено с django в affected: {len(django_in_id)}")
        if django_in_id:
            for fname, pkgs in django_in_id[:5]:
                print(f"  {fname}: {pkgs}")

    # ===== СБОРКА DATAFRAME =====
    print("\n" + "=" * 60)
    print("СБОРКА: ЧИТАЕМ ВСЕ JSON В DATAFRAME")
    print("=" * 60)

    all_data = []
    errors = 0

    for i, filename in enumerate(files):
        if i % 2000 == 0:
            print(f"  Обработано {i}/{len(files)}...")

        try:
            with zf.open(filename) as f:
                vuln = json.loads(f.read())
            all_data.append(vuln)
        except Exception as e:
            errors += 1

    print(f"\n✅ Готово! Загружено: {len(all_data):,} уязвимостей")
    print(f"❌ Ошибок чтения: {errors}")

    # Показываем разнообразие ID
    id_prefixes = {}
    for vuln in all_data:
        prefix = vuln['id'].split('-')[0]
        id_prefixes[prefix] = id_prefixes.get(prefix, 0) + 1

    print(f"\n📊 Типы ID:")
    for prefix, count in sorted(id_prefixes.items()):
        print(f"  {prefix}: {count:,}")

    # Сохраняем в переменную для дальнейшего использования
    print(f"\n📦 Данные готовы в переменной 'all_data'")
    print(f"   Ключи первой записи: {list(all_data[0].keys())}")

Скачиваем https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip (23 МБ)...
Загружено: 23,215,377 байт

ДИАГНОСТИКА: ИЩЕМ DJANGO
Всего файлов в архиве: 19,564
Файлов с 'django' в названии: 0
Из них .json: 0

Странно, django не найден. Пробуем поиск по ID...
В первых 500 файлах найдено с django в affected: 20
  GHSA-2655-q453-22f9.json: ['django', 'django']
  GHSA-287q-jfcp-9vhv.json: ['django-photologue']
  GHSA-296w-6qhq-gf92.json: ['django', 'django', 'django']
  GHSA-2f9x-5v75-3qv4.json: ['django', 'django', 'django']
  GHSA-2gr8-3wc7-xhj3.json: ['social-auth-app-django']

СБОРКА: ЧИТАЕМ ВСЕ JSON В DATAFRAME
  Обработано 0/19564...
  Обработано 2000/19564...
  Обработано 4000/19564...
  Обработано 6000/19564...
  Обработано 8000/19564...
  Обработано 10000/19564...
  Обработано 12000/19564...
  Обработано 14000/19564...
  Обработано 16000/19564...
  Обработано 18000/19564...

✅ Готово! Загружено: 19,564 уязвимостей
❌ Ошибок чтения: 0

📊 Типы ID:
  GHSA: 5,088
  MAL: 11,156


In [ ]:
import pandas as pd
import numpy as np

def normalize_osv_data(data):
    """
    Превращает сырые OSV JSON-записи в две плоские таблицы:
    - vulns_df: одна строка = одна уязвимость
    - affected_df: одна строка = один затронутый пакет
    """
    vulns_list = []
    affected_list = []

    for vuln in data:
        # === Извлечение severity (как в прошлый раз) ===
        db_specific = vuln.get('database_specific', {})
        if not isinstance(db_specific, dict):
            db_specific = {}

        severity_score = np.nan
        severity_string = ''

        # Способ 1: прямой CVSS
        for sev in vuln.get('severity', []):
            if sev.get('type') == 'CVSS_V3':
                try:
                    severity_score = float(sev.get('score', 0))
                except:
                    pass

        # Способ 2: GHSA severity строка -> число
        if pd.isna(severity_score):
            severity_string = db_specific.get('severity', '')
            severity_map = {
                'CRITICAL': 9.5, 'HIGH': 7.5,
                'MODERATE': 5.5, 'MEDIUM': 5.5, 'LOW': 2.5
            }
            severity_score = severity_map.get(severity_string.upper(), np.nan)

        # === Таблица уязвимостей ===
        vuln_record = {
            'id': vuln.get('id'),
            'schema_version': vuln.get('schema_version'),
            'published': vuln.get('published'),
            'modified': vuln.get('modified'),
            'withdrawn': vuln.get('withdrawn'),
            'summary': vuln.get('summary', '').strip() if vuln.get('summary') else '',
            'details': vuln.get('details', '').strip() if vuln.get('details') else '',
            'aliases': '|'.join(vuln.get('aliases', [])),
            'related': '|'.join(vuln.get('related', [])),
            'upstream': '|'.join(vuln.get('upstream', [])),
            'references_count': len(vuln.get('references', [])),
            'credits_count': len(vuln.get('credits', [])),
            'cvss_score': severity_score,
            'severity_string': severity_string,
            'cwe_ids': '|'.join(db_specific.get('cwe_ids', [])) if isinstance(db_specific.get('cwe_ids'), list) else '',
            'github_reviewed': db_specific.get('github_reviewed', False),
            'github_reviewed_at': db_specific.get('github_reviewed_at', ''),
            'nvd_published_at': db_specific.get('nvd_published_at', ''),
            'affected_count': len(vuln.get('affected', [])),
        }

        # Категория severity
        if pd.isna(severity_score):
            vuln_record['severity_category'] = 'UNKNOWN'
        elif severity_score >= 9.0:
            vuln_record['severity_category'] = 'CRITICAL'
        elif severity_score >= 7.0:
            vuln_record['severity_category'] = 'HIGH'
        elif severity_score >= 4.0:
            vuln_record['severity_category'] = 'MEDIUM'
        elif severity_score >= 0.1:
            vuln_record['severity_category'] = 'LOW'
        else:
            vuln_record['severity_category'] = 'NONE'

        # Проверка на fix
        has_any_fix = False
        for aff in vuln.get('affected', []):
            for rng in aff.get('ranges', []):
                for evt in rng.get('events', []):
                    if 'fixed' in evt and evt['fixed']:
                        has_any_fix = True
                        break
        vuln_record['has_any_fix'] = has_any_fix

        vulns_list.append(vuln_record)

        # === Таблица затронутых пакетов ===
        for aff_idx, aff in enumerate(vuln.get('affected', [])):
            pkg = aff.get('package', {})

            aff_record = {
                'vulnerability_id': vuln.get('id'),
                'package_ecosystem': pkg.get('ecosystem', ''),
                'package_name': pkg.get('name', ''),
                'package_purl': pkg.get('purl', ''),
                'affected_index': aff_idx,
            }

            # Есть ли fix в этом affected
            has_fix = False
            fix_versions = []
            for rng in aff.get('ranges', []):
                for evt in rng.get('events', []):
                    if 'fixed' in evt and evt['fixed']:
                        has_fix = True
                        fix_versions.append(evt['fixed'])

            aff_record['has_fix'] = has_fix
            aff_record['fix_versions'] = '|'.join(fix_versions) if fix_versions else None
            aff_record['versions_count'] = len(aff.get('versions', []))

            affected_list.append(aff_record)

    # Сборка DataFrame
    vulns_df = pd.DataFrame(vulns_list)
    affected_df = pd.DataFrame(affected_list)

    # Конвертация дат
    for col in ['published', 'modified', 'withdrawn', 'nvd_published_at', 'github_reviewed_at']:
        if col in vulns_df.columns:
            vulns_df[col] = pd.to_datetime(vulns_df[col], format='mixed', errors='coerce', utc=True)

    # Вычисляемые поля
    vulns_df['days_to_modify'] = (vulns_df['modified'] - vulns_df['published']).dt.days
    vulns_df['is_withdrawn'] = vulns_df['withdrawn'].notna()
    vulns_df['publication_year'] = vulns_df['published'].dt.year
    vulns_df['publication_month'] = vulns_df['published'].dt.month

    # Источник (по префиксу ID)
    vulns_df['source'] = vulns_df['id'].str.split('-').str[0]

    return vulns_df, affected_df

# === ЗАПУСК ===
print("Нормализация данных...")
vulns_df, affected_df = normalize_osv_data(all_data)

print(f"\n✅ Готово!")
print(f"  vulns_df: {vulns_df.shape[0]:,} строк, {vulns_df.shape[1]} колонок")
print(f"  affected_df: {affected_df.shape[0]:,} строк, {affected_df.shape[1]} колонок")

print(f"\n📊 Первые 5 строк vulns_df:")
display(vulns_df[['id', 'schema_version', 'published', 'modified', 'withdrawn', 'summary',
       'details', 'aliases', 'related', 'upstream', 'references_count',
       'credits_count', 'cvss_score', 'severity_string']].head())

display(vulns_df[['cwe_ids', 'github_reviewed', 'github_reviewed_at', 'nvd_published_at',
       'affected_count', 'severity_category', 'has_any_fix', 'days_to_modify',
       'is_withdrawn', 'publication_year', 'publication_month', 'source']].head())


print(f"\n📊 Источники (source):")
print(vulns_df['source'].value_counts())

print(f"\n📊 Severity категории:")
print(vulns_df['severity_category'].value_counts())

print(f"\n📊 Доля с CVSS: {vulns_df['cvss_score'].notna().mean()*100:.1f}%")
print(f"📊 Доля с fix: {vulns_df['has_any_fix'].mean()*100:.1f}%")

Нормализация данных...

✅ Готово!
  vulns_df: 19,564 строк, 26 колонок
  affected_df: 24,900 строк, 8 колонок

📊 Первые 5 строк vulns_df:


,id,schema_version,published,modified,withdrawn,summary,details,aliases,related,upstream,references_count,credits_count,cvss_score,severity_string
0,GHSA-227r-w5j2-6243,1.7.3,2025-03-20 12:32:41+00:00,2025-10-16 07:56:39.452480+00:00,NaT,InvokeAI Arbitrary File Deletion vulnerability,"In invoke-ai/invokeai version v5.0.2, the web ...",CVE-2024-11042,,,4,0,9.5,CRITICAL
1,GHSA-22c2-9gwg-mj59,1.7.3,2025-05-20 18:01:52+00:00,2025-05-20 21:35:29.649435+00:00,NaT,Langroid has a Code Injection vulnerability in...,### Summary\n[LanceDocChatAgent](https://githu...,CVE-2025-46725,,,4,0,7.5,HIGH
2,GHSA-22cc-w7xm-rfhx,1.7.3,2024-02-28 21:30:20+00:00,2024-12-06 05:36:01.844308+00:00,NaT,Mezzanine allows attackers to bypass access co...,An issue in Mezzanine v6.0.0 allows attackers ...,CVE-2024-25170,,,5,0,5.5,MODERATE
3,GHSA-22fp-mf44-f2mq,1.7.3,2025-04-18 20:24:07+00:00,2026-02-04 02:44:06.010018+00:00,NaT,youtube-dl vulnerable to file system modificat...,#### Description\nThis advisory follows the se...,,CVE-2024-38519,,7,0,7.5,HIGH
4,GHSA-22gh-3r9q-xf38,1.7.5,2021-09-20 19:53:30+00:00,2026-03-13 22:14:20.368374+00:00,NaT,Lacking Protection against HTTP Request Smuggl...,"### Impact\n\nIn mitmproxy 7.0.2 and below, a ...",CVE-2021-39214|PYSEC-2021-328,CVE-2021-39214,,4,0,9.5,CRITICAL


,cwe_ids,github_reviewed,github_reviewed_at,nvd_published_at,affected_count,severity_category,has_any_fix,days_to_modify,is_withdrawn,publication_year,publication_month,source
0,CWE-20|CWE-22|CWE-73,True,2025-03-21 16:32:50+00:00,2025-03-20 10:15:23+00:00,1,CRITICAL,True,209.0,False,2025,3,GHSA
1,CWE-94,True,2025-05-20 18:01:52+00:00,2025-05-20 18:15:46+00:00,1,HIGH,True,0.0,False,2025,5,GHSA
2,,True,2024-02-28 22:58:40+00:00,2024-02-28 20:15:41+00:00,1,MEDIUM,False,281.0,False,2024,2,GHSA
3,CWE-434|CWE-669,True,2025-04-18 20:24:07+00:00,NaT,1,HIGH,False,291.0,False,2025,4,GHSA
4,CWE-444,True,2021-09-17 18:30:53+00:00,2021-09-16 15:15:00+00:00,1,CRITICAL,True,1635.0,False,2021,9,GHSA



📊 Источники (source):
source
MAL      11156
GHSA      5088
PYSEC     3312
OSV          8
Name: count, dtype: int64

📊 Severity категории:
severity_category
UNKNOWN     14476
MEDIUM       2071
HIGH         1949
CRITICAL      633
LOW           435
Name: count, dtype: int64

📊 Доля с CVSS: 26.0%
📊 Доля с fix: 37.9%


In [ ]:
# Сводная таблица: source × severity_category
crosstab = pd.crosstab(vulns_df['source'], vulns_df['severity_category'])
print("Таблица сопряжения: source × severity_category")
print(crosstab)

Таблица сопряжения: source × severity_category
severity_category  CRITICAL  HIGH  LOW  MEDIUM  UNKNOWN
source                                                 
GHSA                    633  1949  435    2071        0
MAL                       0     0    0       0    11156
OSV                       0     0    0       0        8
PYSEC                     0     0    0       0     3312


In [ ]:
# Берём только MAL-записи
mal = vulns_df[vulns_df['source'] == 'MAL']

print("=" * 50)
print("MAL — ВРЕДОНОСНЫЕ ПАКЕТЫ")
print("=" * 50)
print(f"Всего записей: {len(mal):,}")
print(f"С fix: {mal['has_any_fix'].sum():,}")
print(f"С отзывом (withdrawn): {mal['is_withdrawn'].sum():,}")
print(f"Среднее references: {mal['references_count'].mean():.1f}")
print()

# По годам
print("По годам:")
print(mal['publication_year'].value_counts().sort_index())
print()

# Примеры summary
print("Примеры summary:")
for s in mal['summary'].sample(10, random_state=1):
    print(f"  • {s}")

MAL — ВРЕДОНОСНЫЕ ПАКЕТЫ
Всего записей: 11,156
С fix: 0
С отзывом (withdrawn): 2
Среднее references: 1.1

По годам:
publication_year
2022      20
2023    6468
2024    2646
2025    1245
2026     777
Name: count, dtype: int64

Примеры summary:
  • Malicious code in libmcpingstudy (PyPI)
  • Malicious code in prometheus-quicker-analysis (PyPI)
  • Malicious code in tpproofreplace (PyPI)
  • Malicious code in httpniggerblackcocksontopfr (PyPI)
  • Malicious code in spiderai (PyPI)
  • Malicious code in toollgtb (PyPI)
  • Malicious code in esqstudyguiping (PyPI)
  • Malicious code in superre (PyPI)
  • Malicious code in mattermost-airflow (PyPI)
  • Malicious code in prof-tg-gdghho-qu (PyPI)


**Здесь видно, что данные, которые пришли с этой базы - просто вредоносные файлы, поэтому для дальнейшего анализа они бесполезны**

In [ ]:
# Берём только PYSEC
pysec = vulns_df[vulns_df['source'] == 'PYSEC']

print("=" * 50)
print("PYSEC — PYPI ADVISORY DATABASE")
print("=" * 50)
print(f"Всего записей: {len(pysec):,}")
print(f"С fix: {pysec['has_any_fix'].sum():,} ({pysec['has_any_fix'].mean()*100:.1f}%)")
print(f"С CVSS: {pysec['cvss_score'].notna().sum():,}")
print(f"Среднее references: {pysec['references_count'].mean():.1f}")
print()

# По годам
print("По годам:")
print(pysec['publication_year'].value_counts().sort_index())
print()

# Есть ли алиасы (связь с CVE/GHSA)?
pysec['has_aliases'] = pysec['aliases'].notna() & (pysec['aliases'] != '')
print(f"Имеют алиасы: {pysec['has_aliases'].sum():,} ({pysec['has_aliases'].mean()*100:.1f}%)")

# Сколько ссылаются на GHSA?
pysec['has_ghsa'] = pysec['aliases'].str.contains('GHSA', na=False)
print(f"Из них с GHSA: {pysec['has_ghsa'].sum():,}")

# Сколько ссылаются на CVE?
pysec['has_cve'] = pysec['aliases'].str.contains('CVE', na=False)
print(f"Из них с CVE: {pysec['has_cve'].sum():,}")

print()
print("Примеры summary:")
for s in pysec['summary'].sample(5, random_state=2):
    print(f"  • {s[:120]}")

PYSEC — PYPI ADVISORY DATABASE
Всего записей: 3,312
С fix: 2,959 (89.3%)
С CVSS: 0
Среднее references: 3.7

По годам:
publication_year
2005      1
2006      8
2007      4
2008     14
2009     13
2010     32
2011     27
2012     41
2013     45
2014    116
2015     42
2016     41
2017    148
2018    154
2019    251
2020    325
2021    889
2022    515
2023    312
2024    259
2025     72
2026      3
Name: count, dtype: int64

Имеют алиасы: 2,983 (90.1%)
Из них с GHSA: 2,870
Из них с CVE: 2,980

Примеры summary:
  • 
  • 
  • 
  • 
  • 


/tmp/ipykernel_14096/3530131030.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pysec['has_aliases'] = pysec['aliases'].notna() & (pysec['aliases'] != '')
/tmp/ipykernel_14096/3530131030.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pysec['has_ghsa'] = pysec['aliases'].str.contains('GHSA', na=False)
/tmp/ipykernel_14096/3530131030.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cav

In [ ]:
# Смотрим все OSV-записи
osv = vulns_df[vulns_df['source'] == 'OSV']

print("=" * 50)
print("OSV — 8 ЗАПИСЕЙ")
print("=" * 50)
for _, row in osv.iterrows():
    print(f"\nID: {row['id']}")
    print(f"  Summary: {str(row['summary'])[:150]}")
    print(f"  Has fix: {row['has_any_fix']}")
    print(f"  Aliases: {row['aliases'][:150] if pd.notna(row['aliases']) else '—'}")
    print(f"  Year: {row['publication_year']}")

OSV — 8 ЗАПИСЕЙ

ID: OSV-2021-1809
  Summary: Heap-buffer-overflow in ujson.cpython-38-x86_64-linux-gnu.so
  Has fix: True
  Aliases: 
  Year: 2022

ID: OSV-2021-449
  Summary: UNKNOWN READ in std::pair<absl::lts_NUMBER_02_25::container_internal::raw_hash_set<absl::lts_NUM
  Has fix: True
  Aliases: 
  Year: 2021

ID: OSV-2021-955
  Summary: Stack-buffer-overflow in Buffer_AppendIndentUnchecked
  Has fix: True
  Aliases: 
  Year: 2021

ID: OSV-2022-1074
  Summary: Invalid-free in _dealloc
  Has fix: True
  Aliases: 
  Year: 2022

ID: OSV-2022-715
  Summary: Segv on unknown address in jpeg_read_scanlines
  Has fix: True
  Aliases: 
  Year: 2022

ID: OSV-2023-885
  Summary: UNKNOWN READ in bytes1_char_at
  Has fix: True
  Aliases: 
  Year: 2023

ID: OSV-2025-409
  Summary: Heap-buffer-overflow in libodbc.so.2
  Has fix: False
  Aliases: 
  Year: 2025

ID: OSV-2025-500
  Summary: UNKNOWN READ in getUShort
  Has fix: True
  Aliases: 
  Year: 2025


In [ ]:
# Выделяем GHSA — чистое ядро уязвимостей
ghsa = vulns_df[vulns_df['source'] == 'GHSA'].copy()

print("=" * 60)
print("GHSA — ОЦЕНЕННЫЕ УЯЗВИМОСТИ (ЧИСТОЕ ЯДРО)")
print("=" * 60)
print(f"Всего: {len(ghsa):,}")
print(f"Период: {int(ghsa['publication_year'].min())} – {int(ghsa['publication_year'].max())}")
print()

# Severity
print("Распределение severity:")
print(ghsa['severity_category'].value_counts())
print()

# Fix rate
print(f"Исправлено: {ghsa['has_any_fix'].sum():,} ({ghsa['has_any_fix'].mean()*100:.1f}%)")
print(f"Отозвано (withdrawn): {ghsa['is_withdrawn'].sum():,}")
print()

# CVSS
print(f"Средний CVSS: {ghsa['cvss_score'].mean():.2f}")
print(f"Медианный CVSS: {ghsa['cvss_score'].median():.1f}")
print(f"Мин / Макс CVSS: {ghsa['cvss_score'].min():.1f} / {ghsa['cvss_score'].max():.1f}")
print()

# Время модификации
print(f"Среднее время модификации: {ghsa['days_to_modify'].mean():.0f} дн.")
print(f"Медианное время модификации: {ghsa['days_to_modify'].median():.0f} дн.")
print()

# Ссылки
print(f"Среднее количество references: {ghsa['references_count'].mean():.1f}")

GHSA — ОЦЕНЕННЫЕ УЯЗВИМОСТИ (ЧИСТОЕ ЯДРО)
Всего: 5,088
Период: 2018 – 2026

Распределение severity:
severity_category
MEDIUM      2071
HIGH        1949
CRITICAL     633
LOW          435
Name: count, dtype: int64

Исправлено: 4,453 (87.5%)
Отозвано (withdrawn): 92

Средний CVSS: 6.51
Медианный CVSS: 7.5
Мин / Макс CVSS: 2.5 / 9.5

Среднее время модификации: 612 дн.
Медианное время модификации: 468 дн.

Среднее количество references: 6.6


In [ ]:
import plotly.graph_objects as go

# Количество уязвимостей по годам
yearly = ghsa.groupby('publication_year').size().reset_index(name='count')
yearly = yearly[yearly['publication_year'].between(2018, 2025)]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=yearly['publication_year'],
    y=yearly['count'],
    mode='lines+markers',
    marker=dict(size=10),
    line=dict(width=3),
    name='Количество'
))

fig.update_layout(
    title="Уязвимости GHSA по годам (PyPI)",
    xaxis_title="Год публикации",
    yaxis_title="Количество уязвимостей",
    height=450
)

fig.show()

In [ ]:
# Количество и средний CVSS по годам
yearly = ghsa.groupby('publication_year').agg(
    count=('id', 'count'),
    avg_cvss=('cvss_score', 'mean')
).reset_index()
yearly = yearly[yearly['publication_year'].between(2018, 2025)]

fig = go.Figure()

# Бар — количество
fig.add_trace(go.Bar(
    x=yearly['publication_year'],
    y=yearly['count'],
    name='Количество уязвимостей',
    marker_color='lightblue',
    yaxis='y1'
))

# Линия — средний CVSS
fig.add_trace(go.Scatter(
    x=yearly['publication_year'],
    y=yearly['avg_cvss'],
    mode='lines+markers',
    name='Средний CVSS',
    marker=dict(size=12, color='red'),
    line=dict(width=3, color='red'),
    yaxis='y2'
))

fig.update_layout(
    title="Количество уязвимостей и средний CVSS по годам",
    xaxis_title="Год публикации",
    yaxis=dict(title="Количество", side='left'),
    yaxis2=dict(title="Средний CVSS", overlaying='y', side='right', range=[0, 10]),
    height=500,
    hovermode='x unified'
)

fig.show()

In [ ]:
import plotly.express as px

# Время модификации по severity (только те, у кого есть дата модификации)
data = ghsa[ghsa['days_to_modify'].notna()].copy()
# Убираем выбросы для наглядности (обрежем по 95-му перцентилю)
cutoff = data['days_to_modify'].quantile(0.95)
data = data[data['days_to_modify'] <= cutoff]

fig = px.box(
    data,
    x='severity_category',
    y='days_to_modify',
    color='severity_category',
    title="Время модификации по severity (без выбросов >95%)",
    labels={'days_to_modify': 'Дней до модификации', 'severity_category': 'Severity'},
    category_orders={'severity_category': ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']}
)

fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# Топ уязвимых пакетов из GHSA
ghsa_ids = ghsa['id'].tolist()
ghsa_affected = affected_df[affected_df['vulnerability_id'].isin(ghsa_ids)]

top = ghsa_affected['package_name'].value_counts().head(15)

print("Топ-15 пакетов по количеству уязвимостей (GHSA):")
for i, (pkg, count) in enumerate(top.items(), 1):
    # Узнаём % исправленных для этого пакета
    pkg_data = ghsa_affected[ghsa_affected['package_name'] == pkg]
    fix_rate = pkg_data['has_fix'].mean() * 100
    print(f"  {i:2d}. {pkg:30s} {count:4d} уязвимостей, исправлено: {fix_rate:.0f}%")

Топ-15 пакетов по количеству уязвимостей (GHSA):
   1. tensorflow                     1324 уязвимостей, исправлено: 100%
   2. tensorflow-gpu                 1311 уязвимостей, исправлено: 100%
   3. tensorflow-cpu                 1307 уязвимостей, исправлено: 100%
   4. django                          403 уязвимостей, исправлено: 100%
   5. salt                            213 уязвимостей, исправлено: 95%
   6. plone                           195 уязвимостей, исправлено: 82%
   7. ansible                         134 уязвимостей, исправлено: 97%
   8. apache-airflow                  115 уязвимостей, исправлено: 100%
   9. open-webui                       97 уязвимостей, исправлено: 73%
  10. apache-superset                  85 уязвимостей, исправлено: 74%
  11. nova                             78 уязвимостей, исправлено: 90%
  12. mlflow                           69 уязвимостей, исправлено: 74%
  13. picklescan                       67 уязвимостей, исправлено: 96%
  14. pillow           

In [ ]:
# Группируем tensorflow-варианты и считаем уникальные уязвимости
import numpy as np

# Выделим tensorflow-семейство
tf_mask = ghsa_affected['package_name'].str.contains('tensorflow', case=False, na=False)
tf_affected = ghsa_affected[tf_mask]
non_tf_affected = ghsa_affected[~tf_mask]

# Считаем уникальные уязвимости
tf_unique_vulns = tf_affected['vulnerability_id'].nunique()
tf_fix_rate = tf_affected['has_fix'].mean() * 100

print(f"TensorFlow (все варианты вместе):")
print(f"  Уникальных уязвимостей: {tf_unique_vulns}")
print(f"  Строк в affected (дубли): {len(tf_affected)}")
print(f"  Исправлено: {tf_fix_rate:.0f}%")
print()

# Топ-10 без дублирования tensorflow
# Сначала заменим имена tensorflow-пакетов на общее
ghsa_affected_clean = ghsa_affected.copy()
ghsa_affected_clean.loc[tf_mask, 'package_name'] = 'tensorflow (все версии)'

top_clean = ghsa_affected_clean['package_name'].value_counts().head(10)

print("Топ-10 пакетов (tensorflow объединён):")
for i, (pkg, count) in enumerate(top_clean.items(), 1):
    pkg_data = ghsa_affected_clean[ghsa_affected_clean['package_name'] == pkg]
    unique = pkg_data['vulnerability_id'].nunique()
    fix_rate = pkg_data['has_fix'].mean() * 100
    print(f"  {i:2d}. {pkg:30s} {unique:4d} уникальных уязвимостей, исправлено: {fix_rate:.0f}%")

TensorFlow (все варианты вместе):
  Уникальных уязвимостей: 435
  Строк в affected (дубли): 3945
  Исправлено: 100%

Топ-10 пакетов (tensorflow объединён):
   1. tensorflow (все версии)         435 уникальных уязвимостей, исправлено: 100%
   2. django                          152 уникальных уязвимостей, исправлено: 100%
   3. salt                             67 уникальных уязвимостей, исправлено: 95%
   4. plone                           100 уникальных уязвимостей, исправлено: 82%
   5. ansible                          66 уникальных уязвимостей, исправлено: 97%
   6. apache-airflow                  111 уникальных уязвимостей, исправлено: 100%
   7. open-webui                       91 уникальных уязвимостей, исправлено: 73%
   8. apache-superset                  68 уникальных уязвимостей, исправлено: 74%
   9. nova                             53 уникальных уязвимостей, исправлено: 90%
  10. mlflow                           67 уникальных уязвимостей, исправлено: 74%


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Подготовка данных
ghsa_ids = ghsa['id'].tolist()
ghsa_affected = affected_df[affected_df['vulnerability_id'].isin(ghsa_ids)].copy()

# Объединяем tensorflow-семейство
tf_mask = ghsa_affected['package_name'].str.contains('tensorflow', case=False, na=False)
ghsa_affected.loc[tf_mask, 'package_name'] = 'tensorflow (все версии)'

# ===== 1. SEVERITY: PIE + BAR =====
fig1 = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'bar'}]],
                     subplot_titles=('Доли', 'Количество'))

sev_counts = ghsa['severity_category'].value_counts()
colors = {'CRITICAL':'#FF0000', 'HIGH':'#FF6B35', 'MEDIUM':'#FFC107', 'LOW':'#4CAF50'}

fig1.add_trace(go.Pie(labels=sev_counts.index, values=sev_counts.values,
                      marker=dict(colors=[colors[c] for c in sev_counts.index]),
                      hole=0.3, textinfo='percent+label'), row=1, col=1)

fig1.add_trace(go.Bar(x=sev_counts.index, y=sev_counts.values,
                      marker=dict(color=[colors[c] for c in sev_counts.index]),
                      text=sev_counts.values, textposition='outside'), row=1, col=2)

fig1.update_layout(title="1. Распределение severity (GHSA)", height=450, showlegend=False)
fig1.show()

# ===== 2. ДИНАМИКА ПО ГОДАМ (количество + средний CVSS) =====
yearly = ghsa.groupby('publication_year').agg(count=('id','count'), avg_cvss=('cvss_score','mean')).reset_index()
yearly = yearly[yearly['publication_year'].between(2018, 2025)]

fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_trace(go.Bar(x=yearly['publication_year'], y=yearly['count'], name='Количество',
                      marker_color='lightblue'), secondary_y=False)
fig2.add_trace(go.Scatter(x=yearly['publication_year'], y=yearly['avg_cvss'], name='Средний CVSS',
                          mode='lines+markers', marker=dict(size=12, color='red'),
                          line=dict(width=3, color='red')), secondary_y=True)
fig2.update_layout(title="2. Уязвимости и средний CVSS по годам", height=450, hovermode='x unified')
fig2.update_yaxes(title_text="Количество", secondary_y=False)
fig2.update_yaxes(title_text="Средний CVSS", range=[0,10], secondary_y=True)
fig2.show()

# ===== 3. ТОП-15 ПАКЕТОВ (уникальные уязвимости) =====
pkg_stats = ghsa_affected.groupby('package_name').agg(
    unique_vulns=('vulnerability_id', 'nunique'),
    fix_rate=('has_fix', 'mean')
).reset_index()
top15 = pkg_stats.nlargest(15, 'unique_vulns')

fig3 = go.Figure(go.Bar(x=top15['unique_vulns'], y=top15['package_name'], orientation='h',
                         marker=dict(color=top15['unique_vulns'], colorscale='Reds'),
                         text=top15['unique_vulns'], textposition='outside'))
fig3.update_layout(title="3. Топ-15 пакетов по уникальным уязвимостям", height=550,
                   yaxis={'categoryorder':'total ascending'}, xaxis_title="Уникальных уязвимостей")
fig3.show()

# ===== 4. BOXPLOT ВРЕМЕНИ МОДИФИКАЦИИ ПО SEVERITY =====
data_box = ghsa[ghsa['days_to_modify'].notna()]
cutoff = data_box['days_to_modify'].quantile(0.95)
data_box = data_box[data_box['days_to_modify'] <= cutoff]

fig4 = px.box(data_box, x='severity_category', y='days_to_modify', color='severity_category',
              title="4. Время модификации по severity (без выбросов >95%)",
              labels={'days_to_modify':'Дней', 'severity_category':'Severity'},
              category_orders={'severity_category':['CRITICAL','HIGH','MEDIUM','LOW']})
fig4.update_layout(height=450, showlegend=False)
fig4.show()

# ===== 5. HEATMAP SEVERITY × ГОД =====
heat = pd.crosstab(ghsa[ghsa['publication_year']>=2018]['publication_year'], ghsa['severity_category'])
fig5 = px.imshow(heat.T, title="5. Тепловая карта severity по годам",
                 labels=dict(x="Год", y="Severity", color="Кол-во"), aspect="auto", color_continuous_scale="YlOrRd")
fig5.update_layout(height=400)
fig5.show()

# ===== 6. FIX RATE ПО SEVERITY (stacked bar) =====
fix_stats = ghsa.groupby('severity_category').agg(total=('id','count'), fixed=('has_any_fix','sum')).reset_index()
fix_stats['unfixed'] = fix_stats['total'] - fix_stats['fixed']
fix_stats['fix_rate'] = (fix_stats['fixed']/fix_stats['total']*100).round(1)

fig6 = go.Figure()
fig6.add_trace(go.Bar(x=fix_stats['severity_category'], y=fix_stats['fixed'], name='Исправлено',
                      marker_color='#4CAF50', text=fix_stats['fix_rate'].apply(lambda x: f'{x}%'), textposition='inside'))
fig6.add_trace(go.Bar(x=fix_stats['severity_category'], y=fix_stats['unfixed'], name='Не исправлено',
                      marker_color='#FF5252'))
fig6.update_layout(title="6. Доля исправленных по severity", height=450, barmode='stack', yaxis_title="Количество")
fig6.show()

# ===== 7. ГИСТОГРАММА ДНЕЙ ДО МОДИФИКАЦИИ =====
mod_data = ghsa[ghsa['days_to_modify'].notna() & (ghsa['days_to_modify']<=1000)]['days_to_modify']
fig7 = go.Figure()
fig7.add_trace(go.Histogram(x=mod_data, nbinsx=50, marker_color='#2196F3', opacity=0.7))
for name, val, color in [('Медиана', mod_data.median(), 'red'), ('Среднее', mod_data.mean(), 'green')]:
    fig7.add_vline(x=val, line_dash="dash", line_color=color, annotation_text=f"{name}: {val:.0f} дн.")
fig7.update_layout(title="7. Распределение времени до модификации", height=450,
                   xaxis_title="Дней", yaxis_title="Количество")
fig7.show()

# ===== 8. СРЕДНИЙ CVSS ПО ПАКЕТАМ (ТОП-15) =====
pkg_cvss = ghsa_affected.merge(ghsa[['id','cvss_score']], left_on='vulnerability_id', right_on='id')
pkg_cvss_agg = pkg_cvss.groupby('package_name').agg(avg_cvss=('cvss_score','mean'), count=('vulnerability_id','nunique')).reset_index()
top_cvss = pkg_cvss_agg[pkg_cvss_agg['count']>=10].nlargest(15, 'avg_cvss')

fig8 = go.Figure(go.Bar(x=top_cvss['avg_cvss'], y=top_cvss['package_name'], orientation='h',
                         marker=dict(color=top_cvss['avg_cvss'], colorscale='Reds'),
                         text=top_cvss['avg_cvss'].round(1), textposition='outside'))
fig8.update_layout(title="8. Топ-15 пакетов по среднему CVSS (мин. 10 уязвимостей)", height=500,
                   yaxis={'categoryorder':'total ascending'}, xaxis_title="Средний CVSS", xaxis_range=[0,10])
fig8.show()

# ===== 9. REFERENCES vs CVSS (scatter) =====
fig9 = px.scatter(ghsa, x='references_count', y='cvss_score', color='severity_category',
                  title="9. Связь количества ссылок и CVSS",
                  labels={'references_count':'Количество ссылок', 'cvss_score':'CVSS'},
                  opacity=0.6, trendline="ols",
                  color_discrete_map={'CRITICAL':'red','HIGH':'orange','MEDIUM':'gold','LOW':'green'})
fig9.update_layout(height=450)
fig9.show()

# ===== 10. WITHDRAWN ПО ГОДАМ =====
w_yearly = ghsa.groupby('publication_year').agg(total=('id','count'), withdrawn=('is_withdrawn','sum')).reset_index()
w_yearly = w_yearly[w_yearly['publication_year']>=2018]
w_yearly['rate'] = w_yearly['withdrawn']/w_yearly['total']*100

fig10 = make_subplots(specs=[[{"secondary_y": True}]])
fig10.add_trace(go.Bar(x=w_yearly['publication_year'], y=w_yearly['total'], name='Всего',
                       marker_color='lightblue'), secondary_y=False)
fig10.add_trace(go.Scatter(x=w_yearly['publication_year'], y=w_yearly['rate'], name='% отозвано',
                           mode='lines+markers', marker=dict(size=10, color='red'),
                           line=dict(width=3, color='red')), secondary_y=True)
fig10.update_layout(title="10. Динамика отзыва уязвимостей", height=450, hovermode='x unified')
fig10.update_yaxes(title_text="Всего", secondary_y=False)
fig10.update_yaxes(title_text="% отозвано", secondary_y=True)
fig10.show()

# ===== 11. ТОП-10 CWE =====
ghsa['primary_cwe'] = ghsa['cwe_ids'].str.split('|').str[0]
cwe_top = ghsa['primary_cwe'].value_counts().head(10)
fig11 = go.Figure(go.Bar(x=cwe_top.values, y=cwe_top.index, orientation='h',
                          marker=dict(color=cwe_top.values, colorscale='Blues'),
                          text=cwe_top.values, textposition='outside'))
fig11.update_layout(title="11. Топ-10 типов уязвимостей (CWE)", height=400,
                    yaxis={'categoryorder':'total ascending'}, xaxis_title="Количество")
fig11.show()

# ===== 12. FIX RATE ПО ПАКЕТАМ (ТОП-15 С ХУДШИМ %) =====
worst_fix = pkg_stats[pkg_stats['unique_vulns']>=10].nsmallest(15, 'fix_rate')
fig12 = go.Figure(go.Bar(x=worst_fix['fix_rate']*100, y=worst_fix['package_name'], orientation='h',
                          marker=dict(color=worst_fix['fix_rate']*100, colorscale='Reds_r'),
                          text=worst_fix['fix_rate'].apply(lambda x: f'{x*100:.0f}%'), textposition='outside'))
fig12.update_layout(title="12. Пакеты с худшим процентом исправлений (мин. 10 уязвимостей)", height=500,
                    yaxis={'categoryorder':'total descending'}, xaxis_title="% исправленных", xaxis_range=[0,100])
fig12.show()

# ===== 13. КАЛЕНДАРНАЯ ТЕПЛОВАЯ КАРТА =====
monthly = ghsa.groupby(['publication_year','publication_month']).size().reset_index(name='count')
monthly_pivot = monthly.pivot(index='publication_month', columns='publication_year', values='count').fillna(0)
fig13 = px.imshow(monthly_pivot, title="13. Календарная тепловая карта публикаций",
                  labels=dict(x="Год", y="Месяц", color="Кол-во"), aspect="auto", color_continuous_scale="Viridis")
fig13.update_layout(height=450)
fig13.show()

# ===== 14. СРЕДНЕЕ ВРЕМЯ МОДИФИКАЦИИ ПО ГОДАМ =====
time_yearly = ghsa.groupby('publication_year')['days_to_modify'].median().reset_index()
time_yearly = time_yearly[time_yearly['publication_year'].between(2018, 2025)]
fig14 = go.Figure(go.Scatter(x=time_yearly['publication_year'], y=time_yearly['days_to_modify'],
                              mode='lines+markers', marker=dict(size=12), line=dict(width=3)))
fig14.update_layout(title="14. Медианное время модификации по годам", height=400,
                    xaxis_title="Год", yaxis_title="Медианное время (дни)")
fig14.show()

# ===== 15. AFFECTED COUNT vs CVSS =====
fig15 = px.scatter(ghsa, x='affected_count', y='cvss_score', color='severity_category',
                   title="15. Количество affected пакетов vs CVSS",
                   labels={'affected_count':'Затронуто пакетов', 'cvss_score':'CVSS'},
                   opacity=0.6, size='references_count', hover_data=['summary'],
                   color_discrete_map={'CRITICAL':'red','HIGH':'orange','MEDIUM':'gold','LOW':'green'})
fig15.update_layout(height=450)
fig15.show()

print("✅ Готово — 15 графиков по чистым данным GHSA!")

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning:

divide by zero encountered in scalar divide

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning:

divide by zero encountered in scalar divide

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning:

divide by zero encountered in scalar divide

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning:

divide by zero encountered in scalar divide



✅ Готово — 15 графиков по чистым данным GHSA!
